In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
import os
import shutil

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


In [ ]:
notebook_login()

In [ ]:
MODEL_ID = "LiquidAI/d1-3B"
HF_USER = "your_hf_username"  # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


In [ ]:

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=800,       # Optimization iterations per block; balances calibration time and accuracy
    group_size=32,   # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,        # Symmetric quantization; enables faster compute and broader engine compatibility
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                      # Target pretrained model to be quantized
    tokenizer=tokenizer,              # Tokenizer used to process text calibration samples
    processor=processor,              # Processor for multimodal models (vision/audio), if present
    scheme="W4A16",                   # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,           # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                     # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                     # Batch size for calibration; kept < 8 to avoid slow execution/high memory overhead
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
    seqlen=2048,                      # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,          # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=False,       # Compiles forward passes using torch.compile to accelerate calibration
    quant_nontext_module=False,       # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
)


In [ ]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # - 'auto_round': Reference AutoRound format (Transformers / native loader)
    format="auto_round",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

In [ ]:
# Files required for remote code execution, tokenization, and processor initialization
auxiliary_files: list[str] = [
    # Custom architecture / remote code execution
    "modeling_d1.py",
    "lfm2_vl.py",
    "hybrid.py",
    "api.py",
    "runner.py",
    "prompt.py",
    "LICENSE",
]

export_subdirs: list[str] = [
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32"),
]

for export_dir in export_subdirs:
    if not os.path.exists(export_dir):
        continue

    print(f"\n--> Injecting inference assets into: {export_dir}")
    for filename in auxiliary_files:
        src = os.path.join(LOCAL_PATH, filename)
        dst = os.path.join(export_dir, filename)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"    Copied: {filename}")
        else:
            print(f"    Warning: Missing {src}")

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    if not os.path.exists(local_dir):
        print(f"[Hub] Directory not found, skipping: {local_dir}")
        return

    full_repo_id = f"{HF_USER}/{repo_name}"
    try:
        api = HfApi()
        print(f"[Hub] Preparing repo: {full_repo_id}...")
        create_repo(
            full_repo_id, repo_type="model", private=False, token=token, exist_ok=True
        )

        print(f"[Hub] Uploading folder from {local_dir}...")
        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] Upload complete: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] Error uploading {full_repo_id}: {e}")


In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()


In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload.")